# Task 1: AlphaMissense 同折基线 —— "我们要打败的那个数"

**目标**: 在 AlphaMissense 分数不为 NaN 的子集 (AM 子集) 上，直接比较:
- AlphaMissense 分数直接作为预测值
- v3 XGBoost 模型在相同行上的 OOF 预测

**核心问题**: 自建模型是否真的比"直接用 AlphaMissense 分数"更好？

In [1]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"

df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")
df_am = pd.read_csv(BASE_PATH + "alphamissense_completed.csv")

assert df_feat["KEY"].is_unique
assert df_am["KEY"].is_unique
assert len(df_feat) == 2179

# Explicit one-to-one merge; never align metadata or scores by row order.
df_feat = df_feat.merge(
    df_am[["KEY", "final_alphamissense_score",
           "final_alphamissense_class", "alphamissense_status"]],
    on="KEY", how="left", validate="one_to_one",
)
assert len(df_feat) == 2179
assert df_feat["Mislocalized"].isin([0, 1]).all()
assert int(df_feat["Mislocalized"].sum()) == 236

# define feature columns
ID_COLS = ["KEY", "Gene", "Mutation_used"]
META_COLS = ["source", "Mislocalized", "phenotype_class", "reloc_v3",
             "final_alphamissense_score", "final_alphamissense_class",
             "alphamissense_status"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + META_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]

X_full = df_feat[feat_cols].values.astype(np.float32)
y_bin = df_feat["Mislocalized"].astype(int).to_numpy()
groups = df_feat["Gene"].values
source_arr = df_feat["source"].values
am_score_arr = df_feat["final_alphamissense_score"].to_numpy(dtype=np.float64)

n_total = len(y_bin)
n_pos = int(y_bin.sum())
n_neg = n_total - n_pos
base_rate = n_pos / n_total

print(f"Data: n={n_total}, pos={n_pos}, neg={n_neg}, base_rate={base_rate:.4f}")
print(f"source: {dict(zip(*np.unique(source_arr, return_counts=True)))}")
print(f"Feature columns: {len(feat_cols)}")

# 5 fold CV
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
print("5 fold CV initialized (StratifiedGroupKFold, n_splits=5, groups=Gene)")

def cv_evaluate_binary(X, y, groups, sample_weight_mode="balanced"):
    """StratifiedGroupKFold, Imputer→Scaler→XGBoost in every fold"""
    xgb_params = dict(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.5,
        objective="binary:logistic", eval_metric="aucpr",
        random_state=42, n_jobs=-1, tree_method="hist",
    )
    oof = np.zeros(len(y), dtype=np.float32)
    per_fold = []
    for fold, (tr_idx, te_idx) in enumerate(cv.split(X, y, groups)):
        X_tr_raw, X_te_raw = X[tr_idx], X[te_idx]
        y_tr = y[tr_idx]
        imp = SimpleImputer(strategy="median")
        scl = StandardScaler()
        X_tr = scl.fit_transform(imp.fit_transform(X_tr_raw)).astype(np.float32)
        X_te = scl.transform(imp.transform(X_te_raw)).astype(np.float32)
        sw = compute_sample_weight("balanced", y_tr)
        model = XGBClassifier(**xgb_params)
        model.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
        oof[te_idx] = model.predict_proba(X_te)[:, 1]
        y_te = y[te_idx]
        per_fold.append({
            "fold": fold,
            "auroc": roc_auc_score(y_te, oof[te_idx]),
            "auprc": average_precision_score(y_te, oof[te_idx]),
            "n": len(y_te), "n_pos": int(y_te.sum())
        })
    return oof, per_fold

def print_metrics(label, y_true, oof, per_fold=None):
    auroc = roc_auc_score(y_true, oof)
    auprc = average_precision_score(y_true, oof)
    n = len(y_true); n_pos = int(y_true.sum()); br = n_pos / n
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  n={n}, pos={n_pos}, base_rate={br:.4f}")
    print(f"  AUROC = {auroc:.4f}")
    print(f"  AUPRC = {auprc:.4f}  (base_rate={br:.4f})")
    if per_fold:
        fa = [f["auroc"] for f in per_fold]
        fp = [f["auprc"] for f in per_fold]
        print(f"  Per-fold AUROC: {[f'{v:.3f}' for v in fa]}  "
              f"mean={np.mean(fa):.4f} ± {np.std(fa):.4f}")
        print(f"  Per-fold AUPRC: {[f'{v:.3f}' for v in fp]}  "
              f"mean={np.mean(fp):.4f} ± {np.std(fp):.4f}")
    return {"label": label, "n": n, "n_pos": n_pos, "base_rate": br,
            "auroc": auroc, "auprc": auprc}

print("CV tools initialized")


Data: n=2179, pos=236, neg=1943, base_rate=0.1083
source: {'additional_benign': np.int64(90), 'main': np.int64(2089)}
Feature columns: 1288
5 fold CV initialized (StratifiedGroupKFold, n_splits=5, groups=Gene)
CV tools initialized


## 1a. AlphaMissense 分数直接作为预测值

不需要训练，分数本身就是对错定位风险的打分。
只在 AM 子集（AlphaMissense score 不为 NaN 的行）上计算。

In [2]:
am_sub = np.isfinite(am_score_arr)
X_am = X_full[am_sub]
y_am = y_bin[am_sub]
g_am = groups[am_sub]
am_v = am_score_arr[am_sub]

n_am = len(y_am)
pos_am = int(y_am.sum())
print(f"AM subset: n={n_am}, pos={pos_am}, base_rate={pos_am/n_am:.4f}")

# AlphaMissense
am_auroc = roc_auc_score(y_am, am_v)
am_auprc = average_precision_score(y_am, am_v)
print(f"\nAlphaMissense-alone:")
print(f"  n={n_am}, pos={pos_am}, base_rate={pos_am/n_am:.4f}")
print(f"  AUROC = {am_auroc:.4f}")
print(f"  AUPRC = {am_auprc:.4f}")


AM subset: n=2140, pos=235, base_rate=0.1098

AlphaMissense-alone:
  n=2140, pos=235, base_rate=0.1098
  AUROC = 0.6491
  AUPRC = 0.1619


## 1b. v3 XGBoost 在 AM 子集上训练+评估

用统一 CV 在 AM 子集内训练 XGBoost，得到 OOF 预测后计算指标。

In [3]:
print("XGBoost on AM subset ...")
oof_am_cv, folds_am_cv = cv_evaluate_binary(X_am, y_am, g_am)
r_am_cv = print_metrics("v3 XGBoost (AM subset CV)", y_am, oof_am_cv, folds_am_cv)


XGBoost on AM subset ...


KeyboardInterrupt: 

## 1c. v3 XGBoost 全量训练，仅在 AM 子集行上评估

这是更公平的比较——模型在整个数据集上训练（充分利用数据），但在完全相同的 AM 子集行上评估。

In [ ]:
print("XGBoost on full dataset ...")
oof_full, folds_full = cv_evaluate_binary(X_full, y_bin, groups)

# only valuate on AM subset
oof_full_on_am = oof_full[am_sub]
r_full_on_am = print_metrics("v3 XGBoost (full dataset training, only evaluating on AM subset)",
                              y_am, oof_full_on_am)


XGBoost on full dataset ...

  v3 XGBoost (full dataset training, only evaluating on AM subset)
  n=2053, pos=234, base_rate=0.1140
  AUROC = 0.5946
  AUPRC = 0.1687  (base_rate=0.1140)


## Task 1 汇总: AlphaMissense vs 模型

**三者在完全相同的 AM 子集行上比较，行完全一致才公平。**

In [5]:
print(f"\n{'='*70}")
print(f"  TASK 1: AlphaMissense vs Model (AM subset, n={n_am}, pos={pos_am})")
print(f"  {'Method':<50s} {'AUROC':>8s} {'AUPRC':>8s}")
print(f"  {'-'*66}")
print(f"  {'AlphaMissense':<50s} {am_auroc:>8.4f} {am_auprc:>8.4f}")
print(f"  {'v3 XGBoost (AM subset CV training)':<50s} {r_am_cv['auroc']:>8.4f} {r_am_cv['auprc']:>8.4f}")
print(f"  {'v3 XGBoost (full dataset training, AM subset evaluation)':<50s} {r_full_on_am['auroc']:>8.4f} {r_full_on_am['auprc']:>8.4f}")
print(f"{'='*70}")

delta = r_full_on_am['auroc'] - am_auroc
if delta > 0:
    print(f"\n✓ 模型 AUROC 比 AlphaMissense 高 {delta:+.4f}")
else:
    print(f"\n✗ 模型 AUROC 比 AlphaMissense 低 {delta:+.4f} —— 模型未能打败 AlphaMissense 基线")
    print(f"  这不是 bug，是如实记录的结论。项目需要更强的特征（如 ΔΔG）来超越这个基线。")



  TASK 1: AlphaMissense vs Model (AM subset, n=2053, pos=234)
  Method                                                AUROC    AUPRC
  ------------------------------------------------------------------
  AlphaMissense                                        0.6362   0.1622
  v3 XGBoost (AM subset CV training)                   0.5914   0.1752
  v3 XGBoost (full dataset training, AM subset evaluation)   0.5946   0.1687

✗ 模型 AUROC 比 AlphaMissense 低 -0.0415 —— 模型未能打败 AlphaMissense 基线
  这不是 bug，是如实记录的结论。项目需要更强的特征（如 ΔΔG）来超越这个基线。
